In [10]:
import openai
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai.api_key = os.getenv('OPENAI_API_KEY')

In [11]:
import openai
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai.api_key = os.getenv('OPENAI_API_KEY')

def get_completion(prompt, model="gpt-3.5-turbo"):
    messages = [{"role": "user", "content": prompt}]
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=0, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message["content"]


In [12]:
%store -r product_name
%store -r product_high_level_requirement
%store -r story_with_ui_mockups
%store -r architecture

print({architecture["system-architecture"]})


Following are the top 3 features of a time tracking app for employees to enter work hours against projects so reports can be generated regarding billable hours and then invoices can be created for the billable hours.

1. Time Entry and Tracking: The app should have an easy-to-use interface that allows employees to enter their work hours quickly and accurately. The app should also have a timer feature that enables employees to track the time they spend on each project.

2. Reporting: The app should be able to generate detailed reports on the time spent on each project and task, including billable and non-billable hours, and total time spent. These reports should be customizable, allowing for filtering and sorting based on different criteria.

3. Invoicing: The app should be able to create invoices for the billable hours worked on each project. The invoices should be customizable and allow for different billing rates and payment terms.




In [75]:
domain_context_prompt = f"""

High level Product Features = ```{product_high_level_requirement} ```
Epics and User Story list for the application named {product_name}: ```{story_with_ui_mockups}```
Software Architecture  = ```{architecture["software-architecture-pattern"]} {architecture["system-architecture"]}```
Application Users of the {product_name}  are: ```Employee, Manager, Accountant```

Your task is to Analyze the Product Features, Epics and Story List and Software Architecrure in detail and generate the following :
    1. Using Domain Driven Design techniques, List the Subdomain, Context Map and Bounded Contexts etc considering all the features and epics.   
"""
domain_context_response = get_completion(domain_context_prompt)
%store architecture

print("\n Domain Context: \n")
print(domain_context_response)

Stored 'architecture' (dict)

 Domain Context: 

Subdomains:
- User Management
- Time Tracking
- Reporting
- Invoicing

Context Map:
- User Management: This subdomain is responsible for managing user accounts and authentication. It interacts with the Time Tracking and Reporting subdomains to ensure that users can only access the features and data that they are authorized to.
- Time Tracking: This subdomain is responsible for allowing employees to enter their work hours against projects and tasks. It interacts with the User Management and Reporting subdomains to ensure that the data entered by employees is accurate and can be used to generate reports and invoices.
- Reporting: This subdomain is responsible for generating reports on the time spent on each project and task. It interacts with the User Management and Time Tracking subdomains to ensure that the reports are accurate and can be customized based on different criteria.
- Invoicing: This subdomain is responsible for creating invo

In [97]:
domain_model_prompt = f"""
Application Users of the {product_name}  are: ```Employee, Manager, Accountant```
Epics and User Story list for the application named {product_name}: ```{story_with_ui_mockups}```
Software Architecture  = ```{architecture["software-architecture-pattern"]} {architecture["system-architecture"]}```
Domain Context  = ```{domain_context_response}```

Your task is to Analyze the Product Epics and Story List and Domain Context and Software Architecture in detail and generate the following :   
    1. Using Domain Driven Design techniques, Identify the Entities, Value Objects with list of attributes and behaviours for each.
    2. List the Services, Repositories etc required for the application. 

"""
domain_model_response = get_completion(domain_model_prompt)


architecture["domain-model"] = domain_model_response
%store architecture
print("\n Domain Model:")
print(domain_model_response)


Stored 'architecture' (dict)

 Domain Model:
1. Entities and Value Objects:
- User: 
  - Attributes: id, name, email, password, role
  - Behaviors: register, login, reset password
- Project:
  - Attributes: id, name, description, client
  - Behaviors: create, update, delete
- Task:
  - Attributes: id, name, description, project
  - Behaviors: create, update, delete
- Time Entry:
  - Attributes: id, user, project, task, start_time, end_time, notes
  - Behaviors: create, update, delete
- Report:
  - Attributes: id, project, task, employee, date range
  - Behaviors: generate, customize
- Invoice:
  - Attributes: id, client, project, billing_rate, payment_terms, status
  - Behaviors: create, customize, preview, send

2. Services and Repositories:
- User Service: 
  - Repositories: UserRepository
  - Services: AuthService, UserService
- Project Service:
  - Repositories: ProjectRepository
  - Services: ProjectService
- Task Service:
  - Repositories: TaskRepository
  - Services: TaskService

In [100]:
domain_diagram_prompt = f"""

Domain Model = ```{domain_model_response}```

Your task is to generate an Entity Relationship PlantUML code from the Domain Model. 
"""
diagram_response = get_completion(domain_diagram_prompt)


architecture["domain-model"] = architecture["domain-model"] + "\n"+ diagram_response

%store architecture
print("\n Domain Model:")
print(diagram_response)


Stored 'architecture' (dict)

 Domain Model:
@startuml

!define ENTITY
!define TABLE(x) class x << (T,#FFAAAA) >>
!define PRIMARY_KEY(x) <u>x</u>
!define UNIQUE(x) <i>x</i>
!define FOREIGN_KEY(x) <b>x</b>

hide methods
skinparam classAttributeIconSize 0

ENTITY User {
  PRIMARY_KEY(id): int
  name: string
  email: string
  password: string
  role: string
}

ENTITY Project {
  PRIMARY_KEY(id): int
  name: string
  description: string
  client: string
}

ENTITY Task {
  PRIMARY_KEY(id): int
  name: string
  description: string
  project: int
}

ENTITY TimeEntry {
  PRIMARY_KEY(id): int
  user: int
  project: int
  task: int
  start_time: datetime
  end_time: datetime
  notes: string
}

ENTITY Report {
  PRIMARY_KEY(id): int
  project: int
  task: int
  employee: int
  date_range: string
}

ENTITY Invoice {
  PRIMARY_KEY(id): int
  client: string
  project: int
  billing_rate: decimal
  payment_terms: string
  status: string
}

User "1" -- "many" TimeEntry
Project "1" -- "many" Task
Proje

In [91]:
%store -r architecture

service_api_endpoints_prompt = f"""
Domain Model = ```{domain_model_response}```

Your task is to Analyze the Entities and Services listed in the Domain Model and generate: 
    1. List of Restful services of the application.
    2. Generate Open API documentation in yaml for the Services.
"""
service_api_endpoints_response = get_completion(service_api_endpoints_prompt)


architecture["service-apis"] = service_api_endpoints_response
%store architecture

print("\n API endpoints: \n")
print(service_api_endpoints_response)

Stored 'architecture' (dict)

 API endpoints: 

1. List of Restful services of the application:
- User:
  - POST /users/register
  - POST /users/login
  - POST /users/reset-password
  - GET /users/{id}
  - PUT /users/{id}
  - DELETE /users/{id}
- Project:
  - POST /projects
  - GET /projects/{id}
  - GET /projects?client={client}
  - PUT /projects/{id}
  - DELETE /projects/{id}
- Task:
  - POST /tasks
  - GET /tasks/{id}
  - GET /tasks?project={project}
  - PUT /tasks/{id}
  - DELETE /tasks/{id}
- Time Entry:
  - POST /time-entries
  - GET /time-entries/{id}
  - GET /time-entries?user={user}&project={project}
  - PUT /time-entries/{id}
  - DELETE /time-entries/{id}
- Report:
  - GET /reports?project={project}&task={task}&employee={employee}&start_date={start_date}&end_date={end_date}
- Invoice:
  - POST /invoices
  - GET /invoices/{id}
  - GET /invoices?client={client}
  - PUT /invoices/{id}
  - DELETE /invoices/{id}
  - POST /invoices/{id}/customize
  - GET /invoices/{id}/preview
  - 